In [29]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn import metrics

# Загрузка данных

In [30]:
%%capture
!wget https://www.dropbox.com/s/64ol9q9ssggz6f1/data_ford_price.xlsx

In [31]:
data = pd.read_excel('data/ford_price.xlsx') 

#  Отбор признаков: мотивация

## Предобработка данных

In [32]:
data = data[['price','year', 'cylinders', 'odometer', 'lat', 'long', 'weather']]
data.dropna(inplace = True)

y = data['price']
x = data.drop(columns='price')

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

## Обучение модели

In [33]:
model = LinearRegression()
model.fit(X_train, y_train)
y_predicted = model.predict(X_test)
 
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

MAE: 4682.957


## Удаление избыточного признака

In [34]:
x.drop('lat', axis = 1, inplace = True)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

In [36]:
model = LinearRegression()
model.fit(X_train, y_train)
y_predicted = model.predict(X_test)
 
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

MAE: 4672.930


#  Отбор признаков: классификация методов

## Метод рекурсивного исключения признаков

Библиотека ***sklearn*** обеспечивает реализацию большинства полезных статистических показателей, например:

коэффициента корреляции Пирсона: ***f_regression***();

дисперсионного анализа ANOVA: ***f_classif***();

хи-квадрата: ***chi2***();

взаимной информации: ***mutual_info_classif***() и ***mutual_info_regression***().

Кроме того, библиотека ***SciPy*** обеспечивает реализацию многих других статистических данных, таких как тау Кендалла (kendalltau) и ранговая корреляция Спирмена (spearmanr).

sklearn также предоставляет множество различных методов фильтрации после расчёта статистики для каждой входной переменной с целевой.

Два наиболее популярных метода:

выбор k лучших переменных: ***SelectKBest***;
выбор переменных верхнего процентиля: ***SelectPercentile***.

In [37]:
from sklearn.feature_selection import RFE

In [38]:
y = data['price']
x = data.drop(columns='price')

In [39]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

In [56]:
from sklearn.metrics import mean_squared_error, r2_score

estimator = LinearRegression()
selector = RFE(estimator, n_features_to_select=3, step=1)
selector = selector.fit(X_train, y_train)
 
selector.get_feature_names_out()

y_pred_test = selector.predict(X_test)

print(f"MSE: {mean_squared_error(y_test, y_pred_test):.2f}")
print(f"R2: {r2_score(y_test, y_pred_test):.2f}")

MSE: 50621735.77
R2: 0.58


In [41]:
X_train.columns

Index(['year', 'cylinders', 'odometer', 'lat', 'long', 'weather'], dtype='str')

In [42]:
selector.ranking_

array([1, 1, 4, 1, 3, 2])

##  МЕТОДЫ ВЫБОРА ПРИЗНАКОВ НА ОСНОВЕ ФИЛЬТРОВ

In [43]:
from sklearn.feature_selection import SelectKBest, f_regression, f_classif

In [44]:
selector = SelectKBest(f_regression, k=3)
selector.fit(X_train, y_train)
 
selector.get_feature_names_out()

array(['year', 'cylinders', 'odometer'], dtype=object)

In [57]:
kbest = SelectKBest(score_func=f_regression, k=3)
kbest.fit(X_train, y_train)

X_train_kbest = kbest.transform(X_train)
X_test_kbest  = kbest.transform(X_test)

lr_kbest = LinearRegression()
lr_kbest.fit(X_train_kbest, y_train)

pred_kbest = lr_kbest.predict(X_test_kbest)
r2_kbest  = r2_score(y_test, pred_kbest)

print(f"MSE: {mean_squared_error(y_test, y_pred_test):.2f}")
print(f"R2: {r2_score(y_test, y_pred_test):.2f}")

MSE: 50621735.77
R2: 0.58
